# Notebook 01: Tracking Pipeline

RF-DETR detection, optional watershed splitting of merged-fly bboxes, OC-SORT tracking, vial assignment, diagnostics, overlay video.

## Stages
1. Configuration
2. (Optional) Background subtraction
3. Vial ROIs
4. RF-DETR + watershed + OC-SORT
5. Vial assignment and ordered IDs
6. Diagnostics
7. Overlay videos

In [1]:
import sys
sys.path.insert(0, "..")

import json
import os
import shutil
from pathlib import Path

import cv2
import pandas as pd
import yaml
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long
from src.roi import draw_and_save_vial_rois, assign_vials_and_ordered_ids
from src.visualization import (
    render_vial_overlay_video,
    render_raw_overlay_video,
    render_detections_video,
)
from utils import (
    load_config,
    make_run_output_dir,
    save_config_snapshot,
    save_run_params,
)

## 1. Configuration

Pick the raw video by its DPE folder and replicate index. Credentials and model id come from `creds_config.yaml`; everything else comes from `config.yaml` via attribute access (`cfg.tracker.confidence`, etc.).

In [4]:
# Pick the video by DPE folder and replicate index.
DPE     = 31
REPLICATE = "005"
RAW_VIDEO = next(Path(f"../data/raw/2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/{DPE} DPE/{REPLICATE}").glob("*-converted.mp4"))

cfg = load_config("../config.yaml")
with open("../creds_config.yaml") as f:
    creds = yaml.safe_load(f)
API_KEY  = creds["API_KEY"]
MODEL_ID = creds["MODEL_ID"]

OUTPUT_PATH = make_run_output_dir(RAW_VIDEO, outputs_root="../outputs")
short_name  = Path(OUTPUT_PATH).name.split("_", 2)[-1]
PATH_TO_VID = str(RAW_VIDEO)

_dest = Path(OUTPUT_PATH) / RAW_VIDEO.name
if not _dest.exists():
    try:
        os.link(RAW_VIDEO, _dest)
    except OSError:
        shutil.copy2(RAW_VIDEO, _dest)

save_config_snapshot(OUTPUT_PATH, config_path="../config.yaml")
_cap = cv2.VideoCapture(str(RAW_VIDEO))
fps = _cap.get(cv2.CAP_PROP_FPS) or cfg.video.fallback_fps
save_run_params(OUTPUT_PATH, "video", {
    "path": str(RAW_VIDEO),
    "fps": fps,
    "width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
})
_cap.release()

## 2. (Optional) Background subtraction and temporal trim

GUI: draw a crop rectangle, pick a `[start, end)` frame range. The `_pp.mp4` output is spatially cropped, temporally trimmed, and background-subtracted (85th-percentile temporal median). All downstream stages run on this clip. Crop params are cached in `roi_library.json` so re-runs skip the GUI.

In [5]:
ROI_LIBRARY = Path("../roi_library.json")
_video_key  = Path(RAW_VIDEO).stem
_library    = json.loads(ROI_LIBRARY.read_text()) if ROI_LIBRARY.exists() else {}

preprocess = True
RAW_CROPPED_VIDEO = None
_crop_params = None

if preprocess:
    pp_out          = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_pp.mp4")
    raw_cropped_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_raw_cropped.mp4")
    _stored_crop = _library.get(_video_key, {}).get("preprocessing") if cfg.roi.use_saved_roi else None

    pp_path, _crop_params = preprocess_bgsub_gui(
        video_path=str(RAW_VIDEO),
        out_mp4=pp_out,
        out_raw_mp4=raw_cropped_out,
        gain=cfg.preprocessing.bg_gain,
        white_level=cfg.preprocessing.bg_white_level,
        bg_sample_stride=cfg.preprocessing.bg_sample_stride,
        bg_percentile=cfg.preprocessing.bg_percentile,
        crop_params=_stored_crop,
    )
    PATH_TO_VID = Path(pp_path)
    RAW_CROPPED_VIDEO = Path(raw_cropped_out)

    _library.setdefault(_video_key, {})["preprocessing"] = _crop_params
    _library[_video_key]["video_path"] = str(RAW_VIDEO)
    ROI_LIBRARY.write_text(json.dumps(_library, indent=2))

    with open(os.path.join(OUTPUT_PATH, "crop_roi.json"), "w") as f:
        json.dump(_crop_params, f, indent=2)

save_run_params(OUTPUT_PATH, "preprocessing", {
    "video_pp": str(PATH_TO_VID),
    "video_raw_cropped": str(RAW_CROPPED_VIDEO) if RAW_CROPPED_VIDEO else None,
    "crop_params": _crop_params,
})

Using stored crop params: x=211, y=101, w=688, h=456, frames=0–388
Saved raw cropped video: ..\outputs\run_132_31DPE_n005\2024-03-01_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_31d_005-converted_raw_cropped.mp4
Saved bgsub video: ..\outputs\run_132_31DPE_n005\2024-03-01_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_31d_005-converted_pp.mp4
Background (85.0th percentile) from 363 frames (stride=1).


## 3. Vial ROIs

Drag a rectangle around each vial; press **q** when done. Cached in `roi_library.json` keyed by video stem so re-runs skip the GUI when `cfg.roi.use_saved_roi` is true.

In [6]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")
_stored_vials = _library.get(_video_key, {}).get("vial_rois")

if cfg.roi.use_saved_roi and _stored_vials is not None:
    _vials = {k: tuple(v) for k, v in _stored_vials.items()}
    with open(ROI_JSON, "w") as f:
        json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
else:
    _vials = draw_and_save_vial_rois(video_path=str(RAW_VIDEO), roi_json_path=ROI_JSON)
    _library.setdefault(_video_key, {})["vial_rois"] = {k: list(v) for k, v in _vials.items()}
    ROI_LIBRARY.write_text(json.dumps(_library, indent=2))

save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

## 4. RF-DETR, watershed split, OC-SORT

RF-DETR detection (cached to `detections_raw.csv` on first run; reused on re-runs). Watershed splits oversized bboxes that contain multiple touching flies before OC-SORT sees them; controlled by `cfg.watershed`. Set `CACHED_DETS` to a prior `detections_raw.csv` to skip inference entirely.

In [7]:
OCSORT_CSV   = os.path.join(OUTPUT_PATH, "ocsort_tracks.csv")
DET_LOG_CSV  = os.path.join(OUTPUT_PATH, "detections_raw.csv")

# Set to a prior run's detections_raw.csv to skip RF-DETR inference.
CACHED_DETS = None
_det_source = CACHED_DETS if (CACHED_DETS and os.path.exists(CACHED_DETS)) else DET_LOG_CSV

t  = cfg.tracker
bw = cfg.tracker.behavioral_weights
df_wide, tracker = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=OCSORT_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    inference_api_url=cfg.roboflow.inference_api_url,
    detection_confidence_rfdetr=t.detection_confidence_rfdetr,
    confidence=t.confidence,
    lost_track_buffer=t.lost_track_buffer,
    minimum_matching_threshold=t.minimum_matching_threshold,
    minimum_consecutive_frames=t.minimum_consecutive_frames,
    asso_func=t.asso_func,
    brownian_pos_noise=t.brownian_pos_noise,
    aspect_weight=t.aspect_weight,
    behavioral_weights=dict(bw),
    overlap_weight_scale=t.overlap_weight_scale,
    inertia=t.inertia,
    delta_t=t.delta_t,
    overlap_iou_scale=t.overlap_iou_scale,
    edge_fraction=t.edge_fraction,
    expected_count=t.expected_count,
    w_under=t.w_under,
    w_over=t.w_over,
    jump_factor=t.jump_factor,
    jump_iou_threshold=t.jump_iou_threshold,
    jump_inertia=t.jump_inertia,
    det_log_csv=_det_source,
    vial_rois=_vials,
    max_frames=None,
    watershed_cfg=dict(cfg.watershed),
)

save_run_params(OUTPUT_PATH, "tracker_output", {
    "ocsort_csv": OCSORT_CSV,
    "frames": int(df_wide.shape[0]),
    "track_count": int(df_wide.shape[1] - 1),
})

with open(os.path.join(OUTPUT_PATH, "tracker_log.json"), "w") as f:
    json.dump({
        "detection_log":     tracker.detection_log,
        "suppressed_tracks": tracker.suppressed_tracks,
        "min_hits":          tracker.min_hits,
        "max_age":           tracker.max_age,
    }, f)

render_detections_video(
    video_path=str(PATH_TO_VID),
    det_log_csv=_det_source,
    out_mp4=os.path.join(OUTPUT_PATH, f"{short_name}_detections_RF-DETR.mp4"),
)

Watershed: median_area=40 px², MAD=5, threshold=55 px², flagged 731 bbox(es) across 320 frame(s).
Watershed debug: wrote 11 success + 50 rejection PNG(s) to ..\outputs\run_132_31DPE_n005\watershed_debug
Saved detection cache: ..\outputs\run_132_31DPE_n005\detections_raw.csv  (11674 detections)
Saved: ..\outputs\run_132_31DPE_n005\ocsort_tracks.csv  (frames=363, tracks=48)
Saved observation logs: ..\outputs\run_132_31DPE_n005\ocsort_tracks_obs_logs.json  (48 trackers)
Saved detections video: ..\outputs\run_132_31DPE_n005\31DPE_n005_detections_RF-DETR.mp4


In [8]:
# Mid-pipeline check: are detections actually reaching the tracker?
# No vial assignment yet, so no per-vial report saved.
run_diagnostics(
    tracker=tracker,
    df_wide=df_wide,
    n_expected=cfg.pipeline.expected_per_vial * len(_vials),
    fps=fps,
    config=cfg,
)

  RUN CONFIGURATION
{
  "roboflow": {
    "model_id": "flies-123/5"
  },
  "video": {
    "fallback_fps": 30
  },
  "calibration": {
    "px_per_cm": 29.0,
    "length_unit": "cm",
    "time_unit": "s"
  },
  "features": {
    "kinematic_three_families": true
  },
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 400,
    "minimum_matching_threshold": 0.2,
    "minimum_consecutive_frames": 1,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weights": {
      "speed": 0.025,
      "scale": 0.025,
      "turning_angle": 0.0,
      "pause": 0.0,
      "acceleration": 0.0
    },
    "overlap_weight_scale": 6.0,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05,
    "inertia": 0.2,
    "delta_t": 3,
    "overlap_iou_scale": 0.1,
    "edge_fraction": 0.1,
    "expected_count": null,
    "w_under": 15.0

## 5. Vial assignment and ordered IDs

Melt the wide OC-SORT CSV to long format, then assign each detection to a vial via the ROI JSON. `ordered_id` is a left-to-right sequential index within each vial.

In [9]:
LONG_CSV    = os.path.join(OUTPUT_PATH, "ocsort_tracks_long.csv")
ORDERED_CSV = os.path.join(OUTPUT_PATH, "ordered_tracks.csv")

with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df = wide_to_long(pd.read_csv(OCSORT_CSV), out_csv=LONG_CSV)
df_ordered = assign_vials_and_ordered_ids(
    ocsort_csv=LONG_CSV,
    roi_json=ROI_JSON,
    out_csv=ORDERED_CSV,
    fps=fps,
)

save_run_params(OUTPUT_PATH, "ordered", {
    "csv": ORDERED_CSV,
    "rows": int(df_ordered.shape[0]),
    "track_count": int(df_ordered["ordered_id"].nunique()),
})
df_ordered.head()

,frame,orig_id,x,y,vial_id,ordered_id,fps
0,0,id1,505.5,418.0,vial5,36,29.892939
1,1,id1,505.0,418.0,vial5,36,29.892939
2,2,id1,505.0,418.0,vial5,36,29.892939
3,3,id1,505.5,418.0,vial5,36,29.892939
4,4,id1,505.0,418.0,vial5,36,29.892939


## 6. Diagnostics

Full diagnostics report (per-vial track counts vs expected, suppressed tracks, re-link impact, coverage histogram). Writes `metrics_report.md` and `metrics_report.html` into the run folder.

In [10]:
run_diagnostics(
    tracker=tracker,
    df_wide=df_wide,
    df_ordered=df_ordered,
    n_expected=cfg.pipeline.expected_per_vial * len(vial_rois),
    fps=fps,
    vial_rois=vial_rois,
    config=cfg,
    output_dir=OUTPUT_PATH,
)

  RUN CONFIGURATION
{
  "roboflow": {
    "model_id": "flies-123/5"
  },
  "video": {
    "fallback_fps": 30
  },
  "calibration": {
    "px_per_cm": 29.0,
    "length_unit": "cm",
    "time_unit": "s"
  },
  "features": {
    "kinematic_three_families": true
  },
  "tracker": {
    "detection_confidence_rfdetr": 0.4,
    "confidence": 0.55,
    "track_activation_threshold": 0.1,
    "lost_track_buffer": 400,
    "minimum_matching_threshold": 0.2,
    "minimum_consecutive_frames": 1,
    "min_area": 20,
    "asso_func": "diou",
    "brownian_pos_noise": 15,
    "aspect_weight": 0.05,
    "behavioral_weights": {
      "speed": 0.025,
      "scale": 0.025,
      "turning_angle": 0.0,
      "pause": 0.0,
      "acceleration": 0.0
    },
    "overlap_weight_scale": 6.0,
    "jump_factor": 2.0,
    "jump_iou_threshold": 0.05,
    "jump_inertia": 0.05,
    "inertia": 0.2,
    "delta_t": 3,
    "overlap_iou_scale": 0.1,
    "edge_fraction": 0.1,
    "expected_count": null,
    "w_under": 15.0

## 7. Overlay videos

Two videos: raw OC-SORT IDs and ordered (within-vial) IDs. Substrate is picked from `cfg.visualization.overlay_source` (`raw_cropped` preferred when preprocessing produced one).

In [11]:
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_ordered.mp4")

_mode = cfg.visualization.overlay_source.lower()
if _mode == "raw_cropped" and RAW_CROPPED_VIDEO is not None:
    OVERLAY_VIDEO = str(RAW_CROPPED_VIDEO)
elif _mode == "raw_cropped":
    OVERLAY_VIDEO = str(RAW_VIDEO)
else:
    OVERLAY_VIDEO = str(PATH_TO_VID)

render_raw_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=LONG_CSV,
    out_mp4=RAW_OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)
render_vial_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=ORDERED_CSV,
    out_mp4=OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)

save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "ordered_overlay": OVERLAY_MP4})
Video(RAW_OVERLAY_MP4, width=800)

  unmatched detections: 229 / 11674
Saved raw overlay video: ..\outputs\run_132_31DPE_n005\31DPE_n005_overlay_raw_ocsort.mp4
  unmatched detections: 229 / 11674
Saved overlay video: ..\outputs\run_132_31DPE_n005\31DPE_n005_overlay_ordered.mp4
